# Exploratory Data Analysis - California Housing

This notebook looks at the cleaned data returned by `src.housing.data.load_dataset()`. It uses the same 20,640 census block groups as the model workflow.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Make the project root importable when the notebook is run from notebooks/.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.housing import data as hd

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

## 2. Load and validate the data

In [ ]:
df = hd.load_dataset()
summary = hd.validate_frame(df)
summary

In [ ]:
df.head()

## 3. Basic structure

In [ ]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Missing values: {int(df.isnull().sum().sum())}")
df.dtypes

In [ ]:
df.describe().T

## 4. Target distribution

In [ ]:
plt.figure()
sns.histplot(df[hd.TARGET_COLUMN], bins=50)
plt.title("Distribution of median house value")
plt.xlabel("Median house value ($100,000)")
plt.ylabel("Count")
plt.show()

The spike at 5.0 is the $500,000 census cap, not a real market limit. Rows at the cap are censored, so their true values may be higher.

In [ ]:
capped = df[hd.TARGET_COLUMN] >= 5.0
print(f"Rows at cap: {capped.sum():,} ({capped.mean():.1%})")

## 5. Feature distributions

In [ ]:
df[hd.EXPECTED_FEATURES].hist(figsize=(12, 8), bins=30)
plt.tight_layout()
plt.show()

## 6. Correlations

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

## 7. Geography

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    df["Longitude"],
    df["Latitude"],
    c=df[hd.TARGET_COLUMN],
    s=5,
    cmap="viridis",
    alpha=0.7,
)
plt.colorbar(scatter, label="Median house value ($100,000)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("House values across California")
plt.show()

## 8. Observations

- `MedInc` has the strongest positive correlation with the target.
- The target is capped at 5.0 ($500,000).
- Higher values cluster along the coast.
- `AveRooms`, `AveBedrms`, and `AveOccup` are per-household averages after the loader conversion.
- 207 `AveBedrms` values are imputed with `AveRooms * 0.5`, so those rows are approximate.
- The data describes 1990 census block groups, not current house prices.